# Research-Grade Multilingual Handwritten OCR Pipeline

This notebook is organized as a debug-first research pipeline for handwritten OCR.

Image Input → Preprocessing → Line Segmentation → Word Segmentation → Word-level Bounding Boxes → Language Detection (per word) → Language-specific OCR → Post-processing

Design priorities:
- Every stage stores intermediate artifacts for inspection.
- Every transformation can be visualized independently.
- Language handling is modular and replaceable.
- Failure cases remain visible instead of being hidden inside a single black-box call.

Use this notebook in top-to-bottom order, or call `run_full_pipeline(...)` directly once the configuration is set.

## Configuration Section

All tunable parameters live here so the pipeline can be debugged and retuned without touching the stage logic.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import math
import re

import cv2
import editdistance
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont

try:
    import pytesseract
except Exception:
    pytesseract = None

try:
    from jiwer import wer as jiwer_wer
except Exception:
    jiwer_wer = None

DEBUG = True
SEED = 42
IMAGE_PATH: Optional[str] = None
OUTPUT_DPI = 120
FIGSIZE_SINGLE = (14, 8)
FIGSIZE_PAIR = (16, 8)

CONFIG: Dict[str, Any] = {
    "debug": DEBUG,
    "seed": SEED,
    "image_path": IMAGE_PATH,
    "output_dpi": OUTPUT_DPI,
    "preprocessing": {
        "clahe_clip_limit": 2.0,
        "clahe_tile_grid_size": (8, 8),
        "gaussian_blur_kernel": (3, 3),
        "denoise_strength": 5,
        "threshold_block_size": 31,
        "threshold_c": 11,
        "morph_kernel": (2, 2),
        "deskew_enabled": True,
    },
    "line_segmentation": {
        "projection_threshold_ratio": 0.06,
        "min_line_height": 10,
        "line_padding": 4,
    },
    "word_segmentation": {
        "min_contour_area": 40,
        "min_word_width": 8,
        "min_word_height": 10,
        "word_gap_factor": 1.35,
        "min_gap_threshold": 12,
        "max_gap_threshold": 80,
        "word_padding": 3,
    },
    "ocr": {
        "english": "eng",
        "marathi": "mar+hin",
        "tamil": "tam",
        "psm": 7,
        "oem": 1,
    },
    "language_map": {
        "Latin": ("English", 0.95),
        "Devanagari": ("Marathi", 0.95),
        "Tamil": ("Tamil", 0.95),
    },
}

np.random.seed(SEED)

@dataclass
class WordResult:
    line_index: int
    word_index: int
    bbox: Tuple[int, int, int, int]
    language: str
    language_confidence: float
    text: str
    ocr_confidence: float
    crop: Optional[np.ndarray] = field(default=None, repr=False)
    language_source: str = "unknown"

    def as_dict(self) -> Dict[str, Any]:
        return {
            "line_index": self.line_index,
            "word_index": self.word_index,
            "bbox": self.bbox,
            "language": self.language,
            "language_confidence": self.language_confidence,
            "text": self.text,
            "ocr_confidence": self.ocr_confidence,
            "language_source": self.language_source,
        }


@dataclass
class PipelineResult:
    original_image: np.ndarray
    debug: Dict[str, Any]
    lines: List[Dict[str, Any]]
    words: List[WordResult]
    reconstructed_text: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "original_image": self.original_image,
            "debug": self.debug,
            "lines": self.lines,
            "words": [word.as_dict() for word in self.words],
            "reconstructed_text": self.reconstructed_text,
        }


print("[OK] Imports and config loaded")
print(f"[OK] DEBUG = {DEBUG}")
print(f"[OK] pytesseract available = {pytesseract is not None}")